# Part 2: Understanding Your LLM

> **Goal:** Build precise intuition for how tokenization, context windows, and model selection affect the systems you can build.

In [ ]:
# Imports

import time
import mlflow
import tiktoken
from chat.client import ChatClient
from utils.tracker import setup_mlflow
from utils.client import DATABRICKS_HOST, DATABRICKS_TOKEN
import pandas as pd
from IPython.display import display
import random
import uuid

## Task 2.1: Tokenization Deep Dive

### Implementation

In [ ]:
prose = (
            "The rapid advancement of artificial intelligence has fundamentally altered "
            "the landscape of modern computing. Machine learning models, particularly "
            "large language models, possess billions of parameters and are trained on "
            "vast corpora of internet text. This extensive training enables them to "
            "perform a wide variety of tasks, from summarizing lengthy documents and "
            "translating between languages to writing functional computer code. However, "
            "these systems are not without their limitations. They can occasionally "
            "generate plausible-sounding but entirely incorrect information, a phenomenon "
            "commonly referred to as hallucination. Furthermore, the immense computational "
            "resources required to train and deploy these models raise important questions "
            "about their environmental impact and the accessibility of state-of-the-art "
            "technology. Researchers are continuously exploring new architectures, such as "
            "more efficient transformer variants, to mitigate these issues. As the field "
            "progresses, the focus is increasingly shifting towards building systems that "
            "are not only highly capable but also robust, interpretable, and aligned with "
            "human values. The next decade will likely witness even more profound changes "
            "as these technologies are integrated more deeply into our daily lives and "
            "critical infrastructure."
        )

code = (
            "def calculate_moving_average(data: list[float], window_size: int) -> list[float]:\n"
            "    \"\"\"\n"
            "    Computes the simple moving average of a list of numbers.\n"
            "    \n"
            "    Args:\n"
            "        data: List of numerical values.\n"
            "        window_size: The number of periods to average over.\n"
            "    \"\"\"\n"
            "    if len(data) < window_size:\n"
            "        return []\n"
            "    \n"
            "    # Calculate initial window sum\n"
            "    window_sum = sum(data[:window_size])\n"
            "    moving_averages = [window_sum / window_size]\n"
            "    \n"
            "    # Slide the window across the rest of the data\n"
            "    for i in range(len(data) - window_size):\n"
            "        window_sum = window_sum - data[i] + data[i + window_size]\n"
            "        moving_averages.append(window_sum / window_size)\n"
            "        \n"
            "    return moving_averages"
        )

json = (
            "{\n"
            "  \"company\": {\n"
            "    \"name\": \"TechCorp\",\n"
            "    \"departments\": [\n"
            "      {\n"
            "        \"name\": \"Engineering\",\n"
            "        \"teams\": {\n"
            "          \"backend\": {\n"
            "            \"lead\": \"Alice\",\n"
            "            \"members\": [\"Bob\", \"Charlie\"]\n"
            "          },\n"
            "          \"frontend\": {\n"
            "            \"lead\": \"David\",\n"
            "            \"members\": [\"Eve\", \"Frank\"]\n"
            "          }\n"
            "        }\n"
            "      }\n"
            "    ]\n"
            "  }\n"
            "}"
        )

md = (
            "# System Architecture\n\n"
            "This document outlines the core components of the system.\n\n"
            "## Requirements\n"
            "* High availability\n"
            "* Low latency\n"
            "* Fault tolerance\n\n"
            "## Deployment\n"
            "To deploy the application, run the following command:\n\n"
            "`docker-compose up --build -d`\n\n"
            "Ensure the `.env` file is properly configured before starting."
        )

math = (
            "The fundamental theorem of calculus connects differentiation and integration.\n"
            "Let $f$ be a continuous real-valued function defined on a closed interval $[a, b]$. "
            "Let $F$ be the function defined, for all $x$ in $[a, b]$, by:\n\n"
            "$$ F(x) = \\int_{a}^{x} f(t) \\, dt $$\n\n"
            "Then $F$ is uniformly continuous on $[a, b]$ and differentiable on the open interval $(a, b)$, "
            "and $F'(x) = f(x)$ for all $x$ in $(a, b)$. Furthermore, if $f$ is Riemann integrable, then:\n\n"
            "$$ \\int_{a}^{b} f(x) \\, dx = F(b) - F(a) $$"
        )

In [ ]:
content_samples = {
        "Prose English": ...,
        "Python code": ...,
        "JSON": ...,
        "Markdown": ...,
        "Mathematical notation": ... 
    }

In [ ]:
setup_mlflow("Task-2.1-Tokenization")

In [ ]:
encodings = {
    ...
}

In [ ]:
results = []

In [ ]:
# Display a summary table of the results.

### Analysis Questions

1. Which content type had the highest characters-per-token ratio (most efficient)? Explain why this makes sense given how subword tokenizers are trained on internet data.
2. Which content type had the lowest ratio (least efficient)? What practical implication does this have for how you'd design prompts that must include this content type?
3. The `cl100k_base` and `o200k_base` encodings use different vocabulary sizes. Did this produce meaningful differences in the ratios you measured? Which tokenizer was more efficient for which content types, and why?
4. Based on your measurements, estimate the maximum number of words of plain English prose that fit in a 4,096-token context window. Show your calculation.

## Task 2.2: Context Window: Needle in a haystack

### Implementation

In [ ]:
enc = tiktoken.get_encoding('cl100k_base')

target_id = ...
num_records = 2500
haystack_lines = []

while len(haystack_lines) < num_records:
    random_id = uuid.uuid4().hex[:8]
    if random_id != target_id:
        record = f"Record-ID: {random_id} | Clearance: {random.randint(1,5)} | Sector: {uuid.uuid4().hex[:6]}\n"
        haystack_lines.append(record)

print(f"No. of tokens in the haystack: {len(enc.encode('\n'.join(haystack_lines)))}")

In [ ]:
setup_mlflow("Task-2.2-Needle-in-a-haystack")

In [ ]:
trials = ...
recall_results = []

client = ChatClient(model="databricks-gemma-3-12b", budgeting=False) 

In [ ]:
target_id = ...
target_sector = ...
needle = ...
question = ...

positions = ...

In [ ]:
# Insert needle into the haystack at positions

# Then run the query in MLflow

# Dont forget to log params and metrics

# Display the results here as a table as well.

### Analysis Questions
 
1. Report your recall results by position (e.g., beginning: 3/3, middle: 1/3, end: 3/3). Is there a positional effect? Is it consistent across your 3 trials per position?
2. Why might a model have structurally lower recall for facts placed at the middle of the context? What does this suggest about how attention is distributed across long sequences?
3. Describe a real system (e.g. a customer support bot, a code assistant, or a document Q&A tool) where such a recall failure would cause a **silent**, hard-to-diagnose bug. What would the failure look like to the end user?
4. Based on your findings, state one concrete design guideline you would apply when constructing context windows for a Robust Information Retrieval system.

## Task 2.3: Model Comparison

### Implementation

In [ ]:
from databricks.sdk import WorkspaceClient

# Initializes client using environment variables (DATABRICKS_HOST and DATABRICKS_TOKEN)
w = WorkspaceClient(host=DATABRICKS_HOST, token=DATABRICKS_TOKEN)
# List all endpoints
endpoints = w.serving_endpoints.list()
# Print endpoint details
for endpoint in endpoints:
    print(f"Name: {endpoint.name} | State: {endpoint.state.ready}")

# > Choose any 3 from Here


In [ ]:
models_to_test = [
        ...
    ]

# For the 3 models you chose, create a mapping of their costs. Look up the costs for a Pay-per-Token scheme in Databricks for them.
cost_rates = {
        ...
    }

In [ ]:
test_query = ...

In [ ]:
setup_mlflow("Task-2.3-Model-Comparison")
benchmark_results = []

In [ ]:
for model_name in models_to_test:
    print(f"\nBenchmarking {model_name}...")
    ...

# Display a summary table of the results here as well. 

### Analysis Questions
 
1. If TTFT is 3× worse for the larger model but quality is only marginally better, describe a specific use case where you would still pick the larger model. Then describe a use case where you would always pick the smaller model regardless of quality.
2. What does TTFT measure that total latency does not? Name a user-facing product feature where TTFT is the more important metric.
3. Your cost proxy is an estimate. Name two real-world pricing factors it ignores that would significantly affect the true cost in production.